# 01 Data collection and provenance

This notebook documents the saved public-source snapshots,
validates the current competitor audit and records the rebuild
order. It uses repository snapshots rather than making live
network requests, so the analysis can be reproduced later.

## 1. Imports and project paths

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, RAW_DIR

## 2. Check the saved source snapshots

The project keeps dated raw files for coworking discovery,
official parish boundaries, census inputs, municipal
validation, points of interest and commercial asking rent.

In [2]:
required_patterns = {
    "OSM coworking snapshot": "osm_coworking_*.json",
    "official parish boundaries": "lisbon_parishes_*.geojson",
    "manual official-site discovery": "manual_discovery_*.csv",
    "targeted competitor audit": "manual_competitor_audit_*.csv",
    "OSM parish POIs": "osm_parish_pois_*.json",
    "commercial rent snapshots": "commercial_rent_listings_*.json",
}

snapshot_check = []
for source, pattern in required_patterns.items():
    matches = sorted(RAW_DIR.glob(pattern))
    snapshot_check.append(
        {
            "source": source,
            "pattern": pattern,
            "files_found": len(matches),
            "latest_file": matches[-1].name if matches else None,
        }
    )

snapshot_check = pd.DataFrame(snapshot_check)
assert snapshot_check["files_found"].gt(0).all()
snapshot_check

,source,pattern,files_found,latest_file
0,OSM coworking snapshot,osm_coworking_*.json,1,osm_coworking_2026-07-24.json
1,official parish boundaries,lisbon_parishes_*.geojson,1,lisbon_parishes_2026-07-24.geojson
2,manual official-site discovery,manual_discovery_*.csv,1,manual_discovery_2026-07-24.csv
3,targeted competitor audit,manual_competitor_audit_*.csv,1,manual_competitor_audit_2026-08-13.csv
4,OSM parish POIs,osm_parish_pois_*.json,1,osm_parish_pois_2026-07-25.json
5,commercial rent snapshots,commercial_rent_listings_*.json,2,commercial_rent_listings_2026-07-30_expansion....


## 3. Validate the current coworking inventory

In [3]:
coworking = pd.read_csv(PROCESSED_DIR / "coworking_locations.csv")
verified = coworking[
    (coworking["active_status"] == "active")
    & (
        coworking["verification_status"]
        == "verified_official_site"
    )
].copy()

assert verified["coworking_id"].is_unique
assert verified[["latitude", "longitude", "parish"]].notna().all().all()
assert verified["source_url"].notna().all()

pd.DataFrame(
    {
        "metric": [
            "verified active locations",
            "parishes with verified supply",
            "pending verification rows",
        ],
        "value": [
            len(verified),
            verified["parish"].nunique(),
            int((coworking["verification_status"] == "pending").sum()),
        ],
    }
)

,metric,value
0,verified active locations,67
1,parishes with verified supply,23
2,pending verification rows,0


## 4. Rebuild order

Live collection scripts are kept in `src/`, but ordinary
reproducibility runs start from the dated raw snapshots. Run
the following steps from the project root:

1. `python src/build_coworking_locations.py --collection-date 2026-07-24`
2. `python src/apply_coworking_review_decisions.py`
3. `python src/apply_coworking_competitor_audit.py --audit-date 2026-08-13`
4. `python src/qa_coworking_locations.py`
5. `python src/build_parish_indicators.py --collection-date 2026-07-25`
6. `python src/qa_parish_indicators.py`
7. Run notebooks `02_cleaning.ipynb` through
   `06_final_charts.ipynb` in order.

This separation preserves the original source evidence and
avoids silently replacing it with newer live responses.

## 5. Collection boundary

Coworking counts represent verified active locations found in
the documented public sources. They are not a complete census.
Missing locations, recent openings and outdated listings remain
possible, so the final output is a screening shortlist for
deeper local due diligence.